# Cyclistic Bike-Share

Coloca `data/processed/cyclistic_trips_clean.csv.gz` o `cyclistic_trips_clean.snappy.parquet` en `data/processed/` y ejecuta en Colab.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')
RECOMPUTE = False
SAVE_ARTIFACTS = True
from pathlib import Path
import pandas as pd, numpy as np, json
ROOT = Path('.')
DATA_PROC = ROOT/'data'/'processed'
DATA_RAW  = ROOT/'data'/'raw'
GRAPHS    = ROOT/'graphs'
for p in [DATA_PROC, DATA_RAW, GRAPHS]: p.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 200)
logging.info('Setup complete')

In [ ]:
parquet = DATA_PROC / 'cyclistic_trips_clean.snappy.parquet'
csvgz   = DATA_PROC / 'cyclistic_trips_clean.csv'
combined= DATA_PROC / 'cyclistic_trips_raw_combined.csv.gz'
df = None
try:
    if parquet.exists():
        df = pd.read_parquet(parquet)
        logging.info(f'Loaded parquet: {parquet}')
    elif csvgz.exists():
        df = pd.read_csv(csvgz, parse_dates=['started_at','ended_at'], low_memory=False)
        logging.info(f'Loaded csv.gz: {csvgz}')
    elif combined.exists():
        df = pd.read_csv(combined, parse_dates=['started_at','ended_at'], low_memory=False)
        logging.info(f'Loaded combined raw csv.gz: {combined}')
    else:
        raise FileNotFoundError('No processed dataset found in data/processed. Sube el archivo o ejecuta la etapa de preparación.')
except Exception as e:
    logging.error('Error al cargar datos: %s', e)
    raise
logging.info(f'Rows loaded: {len(df):,}  Cols: {df.shape[1]}')

In [ ]:
from datetime import datetime, timezone
snapshot = {'read_at': datetime.utcnow().replace(tzinfo=timezone.utc).isoformat(), 'n_rows': int(len(df)), 'n_cols': int(df.shape[1]), 'columns': list(df.columns)}
missing = df.isna().sum().sort_values(ascending=False)
pct_missing = (missing / len(df) * 100).round(3)
missing_summary = pd.DataFrame({'n_missing': missing, 'pct_missing': pct_missing})
missing_summary.to_csv(DATA_PROC / 'missing_summary_before.csv')
with open(DATA_PROC / 'snapshot_before.json','w', encoding='utf-8') as f:
    json.dump(snapshot, f, indent=2)
logging.info('Saved missing_summary_before.csv and snapshot_before.json')

/tmp/ipython-input-203505973.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  snapshot = {'read_at': datetime.utcnow().replace(tzinfo=timezone.utc).isoformat(), 'n_rows': int(len(df)), 'n_cols': int(df.shape[1]), 'columns': list(df.columns)}


In [ ]:
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
if 'member_casual' in df.columns: df['member_casual'] = df['member_casual'].astype(str).str.lower().str.strip()
for dt in ['started_at','ended_at']:
    if dt in df.columns: df[dt] = pd.to_datetime(df[dt], errors='coerce')
if 'started_at' in df.columns and 'ended_at' in df.columns:
    df['ride_length_s'] = (df['ended_at'] - df['started_at']).dt.total_seconds()
    df['ride_length_min'] = df['ride_length_s'] / 60.0
    df['date'] = df['started_at'].dt.date
    df['hour_of_day'] = df['started_at'].dt.hour
    df['day_of_week'] = df['started_at'].dt.day_name()
    df['month'] = df['started_at'].dt.month_name()
    df['is_weekend'] = df['day_of_week'].isin(['Saturday','Sunday'])
else:
    df['ride_length_s'] = np.nan
    df['ride_length_min'] = np.nan
    df['hour_of_day'] = np.nan
    df['day_of_week'] = np.nan
    df['is_weekend'] = False
for col in ['start_lat','start_lng','end_lat','end_lng']:
    df[col] = pd.to_numeric(df.get(col, np.nan), errors='coerce')
df['coords_start_invalid'] = df['start_lat'].isna() | df['start_lng'].isna() | ((df['start_lat']==0) & (df['start_lng']==0))
df['coords_end_invalid']   = df['end_lat'].isna()   | df['end_lng'].isna()   | ((df['end_lat']==0)   & (df['end_lng']==0))
df['distance_km'] = np.nan
valid = ~(df['coords_start_invalid'] | df['coords_end_invalid'])
if valid.any():
    lat1 = np.radians(df.loc[valid,'start_lat'].astype(float).to_numpy())
    lon1 = np.radians(df.loc[valid,'start_lng'].astype(float).to_numpy())
    lat2 = np.radians(df.loc[valid,'end_lat'].astype(float).to_numpy())
    lon2 = np.radians(df.loc[valid,'end_lng'].astype(float).to_numpy())
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a)); R = 6371.0
    df.loc[valid,'distance_km'] = R * c
df.head(3).to_csv(DATA_PROC / 'before_clean_sample.csv', index=False)
logging.info('Preprocessing done; before_clean_sample.csv saved')

In [ ]:
df['long_trip_flag'] = df['ride_length_s'] > 24*3600
mask_invalid_time = df['started_at'].isna() | df['ended_at'].isna()
mask_nonpositive  = df['ride_length_s'] <= 0
rows_to_drop = mask_invalid_time | mask_nonpositive
if rows_to_drop.any():
    df.loc[rows_to_drop].to_csv(DATA_PROC / 'dropped_rows_before_strict.csv', index=False)
df_clean = df[~rows_to_drop].copy()
missing_compare = pd.DataFrame({'before_pct': (df.isna().sum()/len(df)*100).round(3), 'after_pct': (df_clean.isna().sum()/len(df_clean)*100).round(3)})
missing_compare.to_csv(DATA_PROC / 'missing_compare_before_after.csv')
if SAVE_ARTIFACTS:
    out = DATA_PROC / 'cyclistic_trips_clean.csv.gz'
    df_clean.to_csv(out, index=False, compression='gzip')
    logging.info(f'Saved clean dataset to {out} (rows={len(df_clean)})')

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
sns.set()
GRAPHS.mkdir(exist_ok=True)
days = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
hours = list(range(24))
pv = df_clean['member_casual'].value_counts().rename_axis('member_type').reset_index(name='counts')
fig, ax = plt.subplots(figsize=(6,3))
sns.barplot(x='member_type', y='counts', data=pv, ax=ax)
ax.set_title('Trips by member type')
fig.savefig(GRAPHS / 'trips_by_member_type.png', dpi=150, bbox_inches='tight')
plt.close(fig)
if df_clean['ride_length_min'].dropna().size > 10:
    clip = df_clean['ride_length_min'].quantile(0.999)
    filtered_df = df_clean[df_clean['ride_length_min']<=clip]
    sample_size = min(len(filtered_df), 200000) # Ensure sample size is not larger than the filtered data
    if sample_size > 0: # Add a check to ensure there are rows to sample from
        sample = filtered_df.sample(n=sample_size, random_state=1)
        fig, ax = plt.subplots(figsize=(8,4))
        sns.histplot(sample, x='ride_length_min', hue='member_casual', bins=200, stat='density', element='step', common_norm=False, log_scale=(False, True), ax=ax)
        ax.set_xscale('log')
        ax.set_xlabel('Ride length (min) - log scale')
        ax.set_title('Ride length distribution (clipped at 99.9pct)')
        fig.savefig(GRAPHS / 'dist_ride_length_logy.png', dpi=150, bbox_inches='tight')
        plt.close(fig)
for user_type in ['member','casual']:
    sub = df_clean[df_clean['member_casual']==user_type].copy()
    if len(sub)==0: continue
    pivot = sub.groupby(['day_of_week','hour_of_day']).size().reset_index(name='n')
    pivot['day_of_week'] = pd.Categorical(pivot['day_of_week'], categories=days, ordered=True)
    mat = pivot.pivot_table(index='day_of_week', columns='hour_of_day', values='n', aggfunc='sum', fill_value=0, observed=False)
    mat = mat.reindex(index=days, columns=hours, fill_value=0)
    if mat.values.sum()>0:
        fig, ax = plt.subplots(figsize=(14,4))
        sns.heatmap(mat, cmap='magma', ax=ax, cbar_kws={'label':'Trips'})
        ax.set_title(f'Heatmap trips - {user_type} (day x hour)')
        fig.savefig(GRAPHS / f'heatmap_day_hour_{user_type}.png', dpi=150, bbox_inches='tight')
        plt.close(fig)
    counts_by_name = sub['start_station_name'].fillna('UNKNOWN').value_counts()
    top_names = counts_by_name[counts_by_name.index!='UNKNOWN'].nlargest(10)
    if top_names.sum()==0 and 'start_station_id' in sub.columns:
        top_ids = sub['start_station_id'].fillna('UNKNOWN').value_counts().nlargest(10)
        fig, ax = plt.subplots(figsize=(8,4))
        top_ids.sort_values().plot(kind='barh', ax=ax)
        ax.set_title(f'Top 10 start station IDs - {user_type} (fallback)')
        fig.savefig(GRAPHS / f'top10_start_id_{user_type}.png', dpi=150, bbox_inches='tight')
        plt.close(fig)
    else:
        fig, ax = plt.subplots(figsize=(8,4))
        top_names.sort_values().plot(kind='barh', ax=ax)
        ax.set_title(f'Top 10 start stations - {user_type}')
        fig.savefig(GRAPHS / f'top10_start_{user_type}.png', dpi=150, bbox_inches='tight')
        plt.close(fig)
logging.info('Visuals saved')

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
cluster_cols = ['ride_length_min','hour_of_day','distance_km']
cluster_df = df_clean[cluster_cols].dropna().copy()
cap = df_clean['ride_length_min'].quantile(0.99)
cluster_df = cluster_df[cluster_df['ride_length_min'] <= cap].reset_index(drop=True)
if len(cluster_df) < 100:
    logging.info('Too few rows for clustering, skipping')
else:
    n_sample = min(50000, len(cluster_df))
    cluster_sample = cluster_df.sample(n=n_sample, random_state=1).reset_index(drop=True)
    scaler = StandardScaler()
    X = scaler.fit_transform(cluster_sample[cluster_cols])
    inertias = {}
    for k in [2,3,4,5]:
        km = KMeans(n_clusters=k, random_state=1, n_init=10).fit(X)
        inertias[k] = float(km.inertia_)
    logging.info('Inertia candidates: %s', inertias)
    k = 3
    km = KMeans(n_clusters=k, random_state=1, n_init=10).fit(X)
    cluster_sample['cluster'] = km.labels_
    centers = scaler.inverse_transform(km.cluster_centers_)
    centers_df = pd.DataFrame(centers, columns=cluster_cols)
    centers_df.index.name = 'cluster'
    if SAVE_ARTIFACTS and len(centers_df)>0:
        centers_df.to_csv(DATA_PROC / 'cluster_centers.csv', index=True)
        cluster_sample.to_csv(DATA_PROC / 'cluster_sample_with_labels.csv.gz', index=False, compression='gzip')
        logging.info('Saved clustering artifacts')
    else:
        logging.warning('No clustering artifacts saved')

In [ ]:
from scipy import stats
results = {}
members_len = df_clean[df_clean['member_casual']=='member']['ride_length_min'].dropna()
casuals_len = df_clean[df_clean['member_casual']=='casual']['ride_length_min'].dropna()
if len(members_len) >= 30 and len(casuals_len) >= 30:
    m = members_len.sample(n=min(len(members_len),100000), random_state=1)
    c = casuals_len.sample(n=min(len(casuals_len),100000), random_state=1)
    stat, p = stats.mannwhitneyu(m, c, alternative='two-sided')
    results['mannwhitney'] = {'stat':float(stat),'p':float(p)}
cont = pd.crosstab(df_clean['rideable_type'].fillna('UNKNOWN'), df_clean['member_casual'].fillna('UNKNOWN'))
if cont.size>1:
    try:
        chi2, p_chi, dof, exp = stats.chi2_contingency(cont)
        results['chi2'] = {'chi2':float(chi2),'p':float(p_chi),'dof':int(dof)}
    except Exception as e:
        results['chi2_error'] = str(e)
if SAVE_ARTIFACTS:
    (DATA_PROC / 'stat_tests_summary.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
logging.info('Saved stat_tests_summary.json')

# 📊 Conclusiones del Análisis de Viajes Cyclistic

### Hallazgos clave
- Los **usuarios "casual"** muestran picos de uso en **fines de semana y tardes**; mientras que los **"members"** concentran sus viajes en **horas laborales y días de semana**.
- La estación **"Streeter Dr & Grand Ave"** es la más popular para usuarios casuales, en contraste con varias estaciones del centro y universidades para members.
- Los viajes casuales son más cortos en distancia pero más largos en duración promedio.
- El clustering KMeans sugiere **3 perfiles principales de uso**:  
  - **Commuters (mañanas laborales, trayectos cortos).**  
  - **Recreativos (fines de semana, distancias largas).**  
  - **Mixtos/erráticos (sin patrón horario definido).**

### Recomendaciones operativas
- 📌 **Conversión a miembros**: ofrecer descuentos de fin de semana para casuales frecuentes (objetivo: 10–15% conversión).  
- 🚲 **Gestión de flota**: reforzar disponibilidad de bicicletas en estaciones costeras/turísticas durante el verano.  
- 📈 **KPIs sugeridos**:
  - % de casuales convertidos en 3 meses.  
  - Ratio de ocupación de estaciones críticas.  
  - Promedio de duración/distancia por segmento.  

### Próximos pasos
1. Integrar datos meteorológicos para analizar el impacto del clima en la demanda.  
2. Simular escenarios de redistribución de bicicletas usando Optimización Matemática.  
3. Desarrollar dashboards interactivos (Streamlit/Power BI).  


# 🚴‍♀️ Cyclistic Bike-Share Portfolio Project

Este proyecto reproduce un análisis real de comportamiento de usuarios en el sistema de bicicletas compartidas **Cyclistic (Chicago)**.  
Forma parte de mi portafolio de **Optimización Matemática y Ciencia de Datos aplicados a Transporte y Movilidad**.

---

## 🎯 Objetivos
- Analizar patrones de uso entre **usuarios miembros** y **usuarios casuales**.
- Identificar estaciones, horarios y comportamientos representativos.
- Proponer **estrategias de conversión** de casuales a miembros basadas en datos.

---

## 📂 Estructura del repositorio
├── cyclistic_portfolio_final_for_github.ipynb # Notebook principal

├── README.md # Presentación del proyecto

├── HOW_TO_RUN.md # Instrucciones de ejecución

├── requirements.txt # Dependencias mínimas

├── graphs/ # Gráficos PNG generados

└── data/processed/ # Artefactos pequeños procesados


---

## 📊 Principales hallazgos
- Los **miembros** concentran viajes en **horas laborales** y estaciones céntricas.  
- Los **casuales** prefieren **fines de semana, áreas turísticas y viajes recreativos**.  
- Existen **tres clusters de uso**: commuters, recreativos y mixtos.  

---

## 🔑 Recomendaciones
- Incentivar la **conversión de casuales** con planes de fin de semana y beneficios turísticos.  
- Optimizar redistribución de flota en **estaciones costeras y parques**.  
- Monitorear KPIs de conversión y ocupación en dashboards.  

---

## ⚙️ Tecnologías
- Python (pandas, matplotlib, seaborn, scikit-learn).  
- Google Colab.  
- Visualizaciones estáticas (PNG).  

---

## 🧩 Próximos pasos
- Integrar datos de clima y eventos.  
- Desarrollar dashboards interactivos.  
- Explorar modelos de predicción de demanda.  


# ⚙️ Cómo ejecutar el proyecto

Este proyecto está diseñado para correr en **Google Colab** sin configuración extra.  

---

## 1️⃣ Preparar entorno
1. Abre el notebook `cyclistic_portfolio_final_for_github.ipynb` en Google Colab.  
2. Asegúrate de que las dependencias estén instaladas:
   ```bash
   pip install -r requirements.txt


## 2️⃣ Dataset

* El dataset preprocesado ya está incluido:

  * data/processed/cyclistic_trips_clean.snappy.parquet

  * Objetos pequeños (missing_summary_before.csv, stat_tests_summary.json, etc.)

* Los datos crudos originales no se incluyen (por tamaño).
Para regenerarlos, descarga los archivos de Divvy (2022) desde:
https://divvy-tripdata.s3.amazonaws.com/index.html

## 3️⃣ Ejecución

Ejecuta las celdas del notebook en orden. Se generarán:

* Visualizaciones en graphs/.

* Objetos analíticos en data/processed/.

## 4️⃣ Resultados

* Gráficos clave (trips_by_member_type.png, heatmaps, top10_stations.png).

* Clustering (cluster_sample_with_labels.csv.gz).

* Resumen de tests estadísticos (stat_tests_summary.json).

## 5️⃣ Personalización

* Ajusta los parámetros de KMeans en la sección Clustering.

* Cambia las rutas en data/processed/ si deseas otro destino.